# 03 — Embeddings y clustering de negociaciones — LockBit

Genera embeddings semánticos de los mensajes de negociación con `qwen3-embedding` (4096 dims),  
aplica UMAP + HDBSCAN para clustering y visualiza las tácticas de extorsión.

Con solo 4,423 mensajes (vs 195k en BlackBasta) se pueden embeber **todos**  
sin muestreo, y el heatmap operador×operador (35×35) sigue siendo legible.

Produce:
- `data/processed/chat_embeddings.npy`
- `data/processed/chats_with_embeddings.parquet`

## 0. Setup

In [ ]:
# "numpy" es la librería de cálculo numérico. La usamos para trabajar con los vectores
# de embeddings, que son arrays multidimensionales de números.
import numpy as np

# "pandas" para manipular tablas de datos (DataFrames)
import pandas as pd

# "matplotlib.pyplot" para crear gráficas y visualizaciones
import matplotlib.pyplot as plt

# "matplotlib.colors" nos da herramientas para trabajar con colores en los gráficos
import matplotlib.colors as mcolors

# "ollama" para conectarnos al servidor de modelos de lenguaje local
import ollama

# "umap" (Uniform Manifold Approximation and Projection) es una técnica de reducción
# dimensional: toma vectores de miles de dimensiones y los comprime a 2D manteniendo
# las relaciones de similitud. Útil para visualizar datos de alta dimensión en un gráfico.
import umap

# "hdbscan" (Hierarchical Density-Based Spatial Clustering) es un algoritmo de clustering
# que agrupa automáticamente puntos cercanos sin necesitar especificar cuántos grupos hay.
# Es especialmente bueno cuando los clusters tienen formas irregulares.
import hdbscan

# "Path" para manejar rutas de archivos
from pathlib import Path

# "tqdm" para mostrar barras de progreso en bucles largos
from tqdm.auto import tqdm

# Definimos las rutas de entrada y salida
PROCESSED      = Path('../data/processed')
CHATS_CLASS    = PROCESSED / 'chats_classified.parquet'   # Mensajes clasificados (notebook 02)
USERS_IN       = PROCESSED / 'users.parquet'              # Datos de operadores
EMBEDDINGS_NPY = PROCESSED / 'chat_embeddings.npy'        # Vectores de embedding (caché)
CHATS_OUT      = PROCESSED / 'chats_with_embeddings.parquet'  # Dataset final enriquecido

# Nombre del modelo de embedding que usaremos.
# Un modelo de embedding convierte texto en vectores numéricos donde textos similares
# quedan cerca en el espacio vectorial y textos diferentes quedan lejos.
EMBED_MODEL = 'qwen3-embedding'

# Cuántos mensajes procesamos a la vez en cada llamada al modelo.
# Un batch de 32 equilibra velocidad y uso de memoria.
BATCH_SIZE  = 32

# Verificamos que existe el archivo de chats clasificados del notebook 02
assert CHATS_CLASS.exists(), f'Falta {CHATS_CLASS} — ejecuta notebook 02'

# Cargamos los datos de entrada
classified = pd.read_parquet(CHATS_CLASS)
users      = pd.read_parquet(USERS_IN)

# Filtramos para quedarnos solo con mensajes que tienen contenido de texto válido
# (descartamos adjuntos y mensajes vacíos, como en el notebook anterior)
chats_emb = classified[classified.content.notna() & (classified.content.str.strip().str.len() > 3)].copy()

print(f'Mensajes a embeber: {len(chats_emb):,}')
print(f'Modelo de embeddings: {EMBED_MODEL}')
print('Setup OK')

## 1. Generación de embeddings

Usa `ollama.embed()` — distinto de `ollama.chat()`.  
Devuelve `resp.embeddings`: lista de vectores de 768 dimensiones.

In [ ]:
# Generamos los embeddings: convertimos cada mensaje de texto en un vector numérico.
# Un embedding es una representación matemática del significado de un texto.
# El modelo ha aprendido que textos similares (ej: "price?" y "how much?") producen
# vectores parecidos, y textos distintos producen vectores alejados.

if EMBEDDINGS_NPY.exists():
    # Si ya calculamos los embeddings antes, los cargamos del caché para ahorrar tiempo.
    # np.load() carga un archivo .npy (formato binario de numpy) directamente como array.
    embeddings = np.load(EMBEDDINGS_NPY)
    print(f'Cargados desde caché: {embeddings.shape}')
else:
    # Si no hay caché, calculamos los embeddings con el modelo (puede tardar varios minutos)
    texts = chats_emb['content'].tolist()

    # Lista donde iremos acumulando todos los vectores calculados
    all_vecs = []

    # Procesamos los mensajes en lotes (batches) para mayor eficiencia.
    # range(0, len(texts), BATCH_SIZE) genera los índices de inicio de cada lote:
    # 0, 32, 64, 96, ... hasta len(texts)
    for i in tqdm(range(0, len(texts), BATCH_SIZE), desc='Embeddings'):
        # Creamos el lote actual: tomamos los mensajes desde i hasta i+BATCH_SIZE
        # str(t)[:500] convierte a texto y limita a 500 caracteres para controlar el coste
        batch = [str(t)[:500] for t in texts[i:i+BATCH_SIZE]]

        # ollama.embed() genera los embeddings para todos los textos del lote a la vez.
        # Es más eficiente que llamar al modelo una vez por mensaje.
        # resp.embeddings es una lista de vectores, uno por texto del lote.
        resp = ollama.embed(model=EMBED_MODEL, input=batch)
        all_vecs.extend(resp.embeddings)

    # Convertimos la lista de listas a un array numpy 2D:
    # filas = mensajes, columnas = dimensiones del vector (4096 para qwen3-embedding)
    # dtype=np.float32 usa flotantes de 32 bits para ahorrar memoria
    embeddings = np.array(all_vecs, dtype=np.float32)

    # Guardamos los embeddings en disco para no tener que recalcularlos
    # np.save() guarda el array en formato binario .npy (muy eficiente)
    np.save(EMBEDDINGS_NPY, embeddings)
    print(f'Embeddings calculados y guardados: {embeddings.shape}')

# Mostramos la forma del array: (número de mensajes, dimensiones por vector)
print(f'Dimensiones: {embeddings.shape[1]}D')

## 2. Reducción dimensional con UMAP

In [ ]:
# Reducimos los embeddings de 4096 dimensiones a solo 2 dimensiones con UMAP.
# Esto hace posible visualizarlos en una gráfica 2D (como un mapa) donde
# los mensajes similares quedan cerca y los diferentes quedan lejos.
print('Reduciendo a 2D con UMAP...')

# Configuramos el algoritmo UMAP con sus parámetros principales:
reducer = umap.UMAP(
    # n_neighbors=15: cuántos vecinos cercanos considera cada punto para construir el mapa.
    # Un valor más alto produce una vista más global, uno más bajo más local.
    n_neighbors=15,

    # min_dist=0.1: qué tan juntos pueden estar los puntos en el mapa 2D.
    # Un valor más pequeño agrupa más los puntos similares.
    min_dist=0.1,

    # metric='cosine': cómo medimos la distancia entre vectores.
    # La distancia coseno mide el ángulo entre vectores, no su longitud,
    # lo que es ideal para comparar textos (mide dirección semántica, no magnitud).
    metric='cosine',

    # random_state=42: semilla aleatoria para reproducibilidad.
    # Con la misma semilla, UMAP produce siempre el mismo resultado.
    # (42 es un número convencional en ciencia de datos, del libro "Guía del Autoestopista Galáctico")
    random_state=42,

    # n_components=2: reducimos a exactamente 2 dimensiones para poder visualizar
    n_components=2
)

# .fit_transform() calcula el modelo UMAP y devuelve las coordenadas 2D.
# Puede tardar unos segundos con ~4000 mensajes.
coords_2d = reducer.fit_transform(embeddings)
print(f'UMAP completado: {coords_2d.shape}')  # (3977, 2) = 3977 puntos con coordenada x e y

# Añadimos las coordenadas UMAP al DataFrame para poder colorearlas según otras variables
chats_emb = chats_emb.copy()
chats_emb['umap_x'] = coords_2d[:, 0]  # Columna 0 = coordenada X
chats_emb['umap_y'] = coords_2d[:, 1]  # Columna 1 = coordenada Y

## 3. Clustering con HDBSCAN

In [ ]:
# Aplicamos HDBSCAN para descubrir automáticamente grupos de mensajes similares.
# A diferencia de k-means (que requiere especificar el número de grupos),
# HDBSCAN detecta clusters de cualquier forma y tamaño automáticamente.
# Los mensajes que no encajan en ningún cluster se marcan como "ruido" (cluster -1).
clusterer = hdbscan.HDBSCAN(
    # min_cluster_size=20: un grupo debe tener al menos 20 mensajes para ser un cluster.
    # Esto evita que ruido o mensajes aislados formen "micro-clusters" sin sentido.
    min_cluster_size=20,

    # min_samples=5: cuántos vecinos cercanos necesita un punto para ser considerado "central".
    # Valores más altos producen clusters más densos y más ruido.
    min_samples=5
)

# .fit_predict() ajusta el modelo y devuelve el número de cluster para cada punto.
# -1 = ruido (no pertenece a ningún cluster)
# 0, 1, 2, ... = clusters numerados
clusters = clusterer.fit_predict(coords_2d)

# Añadimos los números de cluster al DataFrame para usarlos en las visualizaciones
chats_emb['cluster'] = clusters

# Contamos cuántos clusters distintos se encontraron (excluyendo el -1 de ruido)
n_clusters = (np.unique(clusters) >= 0).sum()

print(f'Clusters encontrados: {n_clusters}')
print(f'Ruido (cluster -1): {(clusters == -1).sum()} mensajes')
print()

# Mostramos cuántos mensajes hay en cada cluster para evaluar su tamaño
print('Tamaño de clusters:')
for c in sorted(np.unique(clusters)):
    # Etiqueta especial para el cluster -1 (mensajes que no encajan en ningún grupo)
    label = 'ruido' if c == -1 else f'cluster {c}'
    print(f'  {label}: {(clusters == c).sum()}')

## 4. Visualizaciones

In [ ]:
# Visualizamos el mapa UMAP coloreado por fase de negociación.
# Cada punto es un mensaje; el color indica en qué fase lo clasificó el LLM.
# Si la clasificación es buena, mensajes de la misma fase deberían agruparse juntos.

# Asignamos un color específico a cada fase del proceso de extorsión
phase_colors = {
    'opening': '#3498db',              # Azul: apertura
    'technical_proof': '#2ecc71',      # Verde: pruebas técnicas
    'price_negotiation': '#e74c3c',    # Rojo: negociación de precio
    'payment_instructions': '#f39c12', # Naranja: instrucciones de pago
    'closing': '#9b59b6',              # Morado: cierre
    'unknown': '#bdc3c7'               # Gris claro: no clasificados
}

# Creamos la figura con tamaño generoso para que los puntos sean visibles
fig, ax = plt.subplots(figsize=(12, 9))

# Dibujamos los puntos fase a fase para poder controlar el color de cada grupo
for phase, color in phase_colors.items():
    # Creamos una máscara booleana: True para los mensajes de esta fase
    mask = chats_emb['phase'] == phase
    if mask.sum() == 0:
        # Si no hay mensajes de esta fase, pasamos a la siguiente
        continue

    # ax.scatter() dibuja puntos en el gráfico usando las coordenadas UMAP
    ax.scatter(
        chats_emb.loc[mask, 'umap_x'],  # Coordenada X de cada punto
        chats_emb.loc[mask, 'umap_y'],  # Coordenada Y de cada punto
        c=color,        # Color del punto
        s=8,            # Tamaño del punto (pequeño para que no se solapen ~4000 puntos)
        alpha=0.6,      # Transparencia: 0=invisible, 1=opaco. 0.6 permite ver puntos superpuestos
        label=phase,    # Etiqueta para la leyenda
        rasterized=True # Renderiza los puntos como imagen para mejor rendimiento
    )

ax.set_title('Mensajes de negociación LockBit — UMAP (coloreado por fase)')
ax.set_xlabel('UMAP 1')
ax.set_ylabel('UMAP 2')

# markerscale=3 hace los símbolos de la leyenda más grandes que los puntos del gráfico
ax.legend(markerscale=3, fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# Segunda visualización UMAP: ahora coloreamos por el rol del remitente
# (operador de LockBit vs víctima) en lugar de por fase.
# Esto nos permite ver si operadores y víctimas usan vocabularios distintos
# que el modelo de embedding ha captado y separado en el espacio 2D.

fig, ax = plt.subplots(figsize=(11, 8))

# Iteramos sobre los dos roles: operador (flag=1) y víctima (flag=0)
# El diccionario mapea el valor de flag a (etiqueta, color)
for flag, (label, color) in {1: ('Operador LockBit', '#e74c3c'), 0: ('Víctima', '#3498db')}.items():
    # Filtramos los mensajes de este remitente
    mask = chats_emb['flag'] == flag

    # Dibujamos los puntos con el color asignado a este rol
    ax.scatter(
        chats_emb.loc[mask, 'umap_x'],
        chats_emb.loc[mask, 'umap_y'],
        c=color,
        s=8,
        alpha=0.5,       # Ligera transparencia para ver la densidad de puntos superpuestos
        label=label,
        rasterized=True
    )

ax.set_title('Mensajes de negociación LockBit — UMAP (rol)')
ax.set_xlabel('UMAP 1')
ax.set_ylabel('UMAP 2')
ax.legend(markerscale=3)
plt.tight_layout()
plt.show()

## 5. Similitud coseno entre operadores

Comparamos el estilo semántico de los operadores usando el centroide  
de sus embeddings. Con 35 operadores, la matriz es legible.

In [ ]:
# Importamos la función para calcular la similitud coseno entre vectores.
# La similitud coseno mide qué tan "parecidos" son dos vectores ignorando su magnitud.
# Devuelve valores entre -1 (opuestos) y 1 (idénticos).
from sklearn.metrics.pairwise import cosine_similarity

# Filtramos solo los mensajes enviados por los operadores de LockBit (flag == 1)
# para comparar los estilos de escritura de cada operador
op_msgs = chats_emb[chats_emb.flag == 1].copy()
op_msgs_idx = op_msgs.index.tolist()
chats_emb_list = chats_emb.index.tolist()

# Creamos un mapa de índice → posición en el array de embeddings.
# Esto es necesario porque el DataFrame puede tener índices no consecutivos (ej: 0, 5, 10...)
# pero el array numpy usa posiciones consecutivas (0, 1, 2...).
# pos_map[idx] devuelve la fila del array embeddings que corresponde al mensaje con ese índice.
pos_map = {idx: i for i, idx in enumerate(chats_emb_list)}
op_msgs['emb_pos'] = op_msgs.index.map(pos_map)

# Calculamos el "centroide" de cada operador: el vector promedio de todos sus mensajes.
# El centroide representa el "estilo medio" del operador en el espacio semántico.
# Si dos operadores tienen centroides similares, escriben de forma parecida.
op_centroids = {}

# Seleccionamos solo los operadores con al menos 5 mensajes para tener una muestra representativa
active_ops = op_msgs.groupby('advid').size()
active_ops = active_ops[active_ops >= 5].index.tolist()

for op_id in active_ops:
    # Obtenemos las posiciones en el array numpy de los mensajes de este operador
    positions = op_msgs[op_msgs.advid == op_id]['emb_pos'].tolist()

    # Seleccionamos esas filas del array de embeddings
    vecs = embeddings[positions]

    # Calculamos la media de todos los vectores (centroide)
    # .mean(axis=0) calcula la media columna a columna (promedio de cada dimensión)
    centroid = vecs.mean(axis=0)

    # Normalizamos el centroide dividiéndolo por su norma (longitud).
    # Esto hace que todos los centroides tengan longitud 1, lo que es necesario para
    # que la similitud coseno sea comparable entre operadores con diferente volumen de mensajes.
    centroid /= np.linalg.norm(centroid)
    op_centroids[op_id] = centroid

print(f'Operadores con centroides: {len(op_centroids)}')

# Construimos la matriz de similitud: una tabla donde cada celda [i,j] contiene
# la similitud entre el operador i y el operador j.
# Si [i,j] es alto (≈1.0), los dos operadores escriben de forma muy parecida.
op_ids = list(op_centroids.keys())
centroid_matrix = np.array([op_centroids[oid] for oid in op_ids])

# cosine_similarity() calcula todas las similitudes de una vez, devolviendo una matriz cuadrada
sim_matrix = cosine_similarity(centroid_matrix)

# Construimos las etiquetas: usamos el nombre de usuario en lugar del ID numérico
# id_to_login es un diccionario {id: login} construido a partir de la tabla users
id_to_login = dict(zip(users.id, users.login))
labels = [id_to_login.get(oid, f'ID_{oid}') for oid in op_ids]

In [ ]:
# Visualizamos la matriz de similitud como un heatmap (mapa de calor).
# Filas y columnas son operadores; el color de cada celda indica cuán similares son.
# Verde = muy similares (≈1.0), rojo = muy diferentes (≈0.5).
# La diagonal siempre es 1.0 (un operador es idéntico a sí mismo).

n = len(labels)

# Ajustamos el tamaño de la figura según el número de operadores.
# max(8, n * 0.35) asegura un mínimo de 8 pulgadas, creciendo con n.
fig_size = max(8, n * 0.35)

fig, ax = plt.subplots(figsize=(fig_size, fig_size))

# Dibujamos el heatmap:
# cmap='RdYlGn' usa la paleta rojo-amarillo-verde (típica para similitudes)
# vmin=0.5, vmax=1.0 fija la escala de colores entre 0.5 y 1.0
# (todos los valores serán ≥0.5 porque los vectores normalizados son positivos)
im = ax.imshow(sim_matrix, cmap='RdYlGn', vmin=0.5, vmax=1.0)

# Configuramos los ejes X e Y con los nombres de los operadores
ax.set_xticks(range(n))
ax.set_yticks(range(n))

# Rotamos 45 grados los nombres del eje X para que no se solapen
# ha='right' alinea el texto a la derecha para que quede bien con la rotación
ax.set_xticklabels(labels, rotation=45, ha='right', fontsize=7)
ax.set_yticklabels(labels, fontsize=7)

# Barra de colores lateral: explica qué color corresponde a qué valor de similitud
# fraction=0.02 hace la barra de colores estrecha para no ocupar mucho espacio
plt.colorbar(im, ax=ax, fraction=0.02, label='Similitud coseno')

ax.set_title('Similitud semántica entre operadores — LockBit\n(1.0 = estilo idéntico)')
plt.tight_layout()
plt.show()

# Identificamos los pares de operadores con el estilo más similar.
# Esto puede indicar que son la misma persona, que usan el mismo script,
# o que tienen el mismo mentor dentro de la organización.
print('Top 5 pares de operadores con estilo más similar:')
pairs = []
for i in range(n):
    # range(i+1, n) empieza en i+1 para no repetir pares (evita (A,B) y (B,A))
    for j in range(i+1, n):
        pairs.append((sim_matrix[i,j], labels[i], labels[j]))

# Ordenamos de mayor a menor similitud
pairs.sort(reverse=True)

for sim, a, b in pairs[:5]:
    # :<20s alinea el nombre en 20 caracteres hacia la izquierda
    print(f'  {a:20s} ↔ {b:20s}  sim={sim:.3f}')

## 6. Guardar

In [ ]:
# Guardamos el dataset enriquecido con las coordenadas UMAP y el número de cluster.
# Este archivo será la entrada del notebook 04 (analista conversacional).
# index=False evita guardar el índice del DataFrame como columna extra.
chats_emb.to_parquet(CHATS_OUT, index=False)

# Mostramos un resumen de lo que se produjo en este notebook
print(f'Embeddings      → {EMBEDDINGS_NPY}  ({embeddings.shape})')
print(f'Dataset enriquecido → {CHATS_OUT}')
print(f'\nResumen:')
print(f'  Mensajes embebidos : {len(chats_emb):,}')
print(f'  Clusters HDBSCAN   : {n_clusters}')
print(f'  Operadores analizados: {len(op_centroids)}')